In [45]:
import psycopg2  
import pandas as pd
from faker import Faker
from random import randint
import random
import datetime
from io import StringIO
import time
import statistics

In [ ]:
#!python -m pip install faker



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
fake = Faker()
nr = 50
students = {}

for i in range(nr):
    students[i] = {
        'name': fake.name(),
        'address': fake.address(),
        'latitude': float(fake.latitude()),
        'longitude': float(fake.longitude()),
        'birth_date':fake.profile()["birthdate"]
    }

students_df=pd.DataFrame.from_dict(students,orient="index")


print(students_df.head())

               name                                            address  \
0     Megan Rollins            2444 Curry Keys\nWest Richard, NJ 12353   
1  Olivia Patterson  5913 Garrett Drive Suite 737\nNatalieborough, ...   
2  Timothy Mcdonald  6762 Jeffery Ridge Suite 354\nEileenport, TX 5...   
3   Stephanie White           0875 Hammond Course\nAlanburgh, FM 63281   
4     Carlos Benson    14686 Kayla Ford Apt. 659\nEast Julie, NE 49814   

    latitude   longitude  birth_date  
0  76.121433 -106.897398  1949-06-18  
1   0.649898  -41.223738  1945-12-19  
2 -74.995079  130.635896  1992-11-09  
3 -20.004509   47.842406  1947-11-16  
4 -44.885330  -45.030118  1958-07-11  


In [28]:
students_list=students_df.values.tolist()
students_list[0:10]

[['Megan Rollins',
  '2444 Curry Keys\nWest Richard, NJ 12353',
  76.121433,
  -106.897398,
  datetime.date(1949, 6, 18)],
 ['Olivia Patterson',
  '5913 Garrett Drive Suite 737\nNatalieborough, IN 81727',
  0.6498975,
  -41.223738,
  datetime.date(1945, 12, 19)],
 ['Timothy Mcdonald',
  '6762 Jeffery Ridge Suite 354\nEileenport, TX 55590',
  -74.995079,
  130.635896,
  datetime.date(1992, 11, 9)],
 ['Stephanie White',
  '0875 Hammond Course\nAlanburgh, FM 63281',
  -20.0045085,
  47.842406,
  datetime.date(1947, 11, 16)],
 ['Carlos Benson',
  '14686 Kayla Ford Apt. 659\nEast Julie, NE 49814',
  -44.88533,
  -45.030118,
  datetime.date(1958, 7, 11)],
 ['Christina Hess',
  '89010 Brown Plaza Suite 679\nNew Brianside, AS 26502',
  86.606223,
  9.844185,
  datetime.date(1965, 12, 9)],
 ['Anthony Cannon MD',
  '16436 Adkins Track\nLake Roberto, VA 03778',
  81.5088505,
  148.20343,
  datetime.date(1992, 2, 22)],
 ['Laura Elliott',
  '652 Powell Radial\nLaurastad, WY 78265',
  6.197802,
  -8

In [5]:
conn=psycopg2.connect(database="sandbox_db",password="jaune2000",host="localhost",port="5432",user="postgres")
cur=conn.cursor()

In [39]:
cur.execute("DROP TABLE IF EXISTS student ")

query=("""CREATE TABLE student(
        id bigint GENERATED ALWAYS AS IDENTITY,
        name varchar(255) NOT NULL,
        address varchar(255),
        latitude NUMERIC(10,8),
        longitude NUMERIC(16,8),
        birth_date DATE
       )
""")
cur.execute(query)

In [38]:
conn.rollback()

In [ ]:

nb_rows=10
num_test=10

In [ ]:
| 1 | `cursor.execute()` dans une boucle `for`, un `commit()` par ligne | la version naïve |

In [46]:
times_list=[]    
for test in range(num_test):
    start=time.perf_counter()
    for i in range(nb_rows):
        cur.execute("INSERT INTO student VALUES(DEFAULT,%s,%s,%s,%s,%s)",students_list[i])
        cur.execute("COMMIT")
    end=time.perf_counter()
    times_list.append(end-start)
print(statistics.median(times_list))

0.004776050000145915


In [ ]:
| 2 | `cursor.execute()` dans une boucle `for`, **un seul** `commit()` à la fin | mesurer l'effet de la transaction unique |

In [49]:
times_list=[]    
for test in range(num_test):
    start=time.perf_counter()
    for i in range(nb_rows):
        cur.execute("INSERT INTO student VALUES(DEFAULT,%s,%s,%s,%s,%s)",students_list[i])
    cur.execute("COMMIT")
    end=time.perf_counter()
    times_list.append(end-start)
print(statistics.median(times_list))

0.0032208000011451077


In [ ]:
| 3 | `cursor.executemany()` | la méthode « évidente » (pourtant assez lente) |

In [53]:
len(students_list)

10

In [ ]:
for test in range(num_test):
    start=time.perf_counter()
    cur.executemany("INSERT INTO student VALUES(%DEFAULT,%s,%s,%s,%s,%s) ",students_list[0:] )
    end=time.perf_counter()
    times_list.append(end-start)
print(statistics.median(times_list))

IndexError: list index out of range

In [ ]:
| 4 | `psycopg2.extras.execute_batch()` (jouez sur `page_size`) | regroupe les requêtes en paquets |

In [ ]:
psycopg2.extras.execute_batch("INSERT INTO student VALUES(%DEFAULT,%s,%s,%s,%s)",page_size=5)

In [ ]:
| 5 | `psycopg2.extras.execute_values()` (jouez sur `page_size`) | un seul `INSERT` à valeurs multiples |

In [ ]:
psycopg2.extras.execute_values("INSERT INTO student VALUES(%DEFAULT,%s,%s,%s,%s)",page_size=5)

In [ ]:
| 6 | `COPY` via `cursor.copy_expert()` + `StringIO` | le chemin rapide de PostgreSQL |

In [22]:

cur.copy_expert("INSERT INTO student VALUES(DEFAULT,%s,%s,%s,%s)",StringIO())

SyntaxError: ERREUR:  erreur de syntaxe sur ou près de « % »
LINE 1: INSERT INTO student VALUES(DEFAULT,%s,%s,%s,%s)
                                           ^


In [ ]:
| 7  | `pandas.to_sql()` par défaut | la facilité (mais pas la plus rapide!!) |

In [ ]:
students_list.to_sql("student",conn)

In [ ]:
| 8  | `pandas.to_sql(method='multi', chunksize=...)` | la variante « groupée » de pandas |

In [ ]:
students_list.to_sql("student",conn,method='multi', chunksize=5)

In [ ]:
| 9 💫 | `pandas.to_sql(method=callable)` branché sur `COPY` | le meilleur des deux mondes |

In [ ]:
students_list.to_sql(method=callable)

In [ ]:
| 10 💫 | `COPY` via un itérateur (sans tout charger en mémoire) | le plus rapide ET le plus sobre en RAM |

In [ ]:

with cur.copy("COPY student (name, address,latitude,longitude,birth_day) FROM STDIN") as cop:
    for student in students_list[0:nb_rows]:
        cop.write_row(student)

AttributeError: 'psycopg2.extensions.cursor' object has no attribute 'copy'